# 01 — Dataset Exploration

Load the processed dataset (Zenodo 77 GHz FMCW or synthetic), inspect class
distribution, sample shapes, and basic signal statistics.

> **Note**: If the real dataset has not been downloaded yet, the notebook
> falls back to the synthetic generator so that every cell can still run.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from radar_drone_vision.datasets.synthetic import SyntheticGenerator
from radar_drone_vision.datasets.manifest import DatasetManifest

## 1. Load or generate dataset

In [ ]:
PROCESSED_DIR = pathlib.Path("../data/processed/zenodo_77ghz")

if PROCESSED_DIR.exists() and (PROCESSED_DIR / "manifest.parquet").exists():
    manifest = DatasetManifest(name="zenodo_77ghz", base_dir=PROCESSED_DIR)
    df = manifest.load()
    print(f"Loaded real dataset: {len(df)} samples")
else:
    print("Real dataset not found — using synthetic data.")
    gen = SyntheticGenerator(seed=42)
    samples = gen.generate_balanced_dataset(n_per_class=200)
    import tempfile
    tmpdir = tempfile.mkdtemp()
    manifest = DatasetManifest(name="synthetic", base_dir=tmpdir)
    df = manifest.build_from_samples(samples)
    print(f"Generated synthetic dataset: {len(df)} samples")

df.head()

## 2. Class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["label"].value_counts().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Class distribution (fine labels)")
axes[0].set_ylabel("Count")

df["label_binary"].map({1: "UAV", 0: "non-UAV"}).value_counts().plot.bar(
    ax=axes[1], color=["#e74c3c", "#2ecc71"]
)
axes[1].set_title("Binary distribution")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 3. Sample shapes and signal statistics

In [ ]:
# Load a few samples and inspect shapes
for i in range(min(5, len(df))):
    npz = manifest.load_sample(i)
    sig = npz["signal"]
    label = npz.get("label_name", npz.get("label", "?"))
    print(f"Sample {i}: shape={sig.shape}, dtype={sig.dtype}, "
          f"min={sig.min():.4f}, max={sig.max():.4f}, label={label}")

## 4. Amplitude histogram per class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for label_bin, name in [(1, "UAV"), (0, "non-UAV")]:
    idxs = df[df["label_binary"] == label_bin].index[:50]
    all_vals = []
    for idx in idxs:
        sig = manifest.load_sample(idx)["signal"]
        all_vals.append(sig.ravel())
    all_vals = np.concatenate(all_vals)
    ax.hist(all_vals, bins=100, alpha=0.5, density=True, label=name)

ax.set_xlabel("Amplitude")
ax.set_ylabel("Density")
ax.set_title("Signal amplitude distribution by class")
ax.legend()
plt.tight_layout()
plt.show()